
# 08 — Final publication robustness: identification, benchmarks, regimes, zero-coupon and finite-sample checks

**Upload three files to the same Colab session, then Run All:**

1. `08_macroanchor-bond-risk-premia.ipynb`
2. `paper_config_monthly_panel.csv`
3. `feds200628.csv`

The notebook validates both inputs against fixed byte-level SHA-256 hashes. It never chooses an
economic series because a website happened to answer.

Three layers are kept separate:

1. **Canonical paper configuration:** the frozen July-2026 FRED GS/CMT panel, the proxy for
   \(r^*\), Kim–Wright `THREEFYTP10`, and the canonical `post_ewma_bfill` anchor.
2. **Identification and benchmark checks:** strict past-only orthogonalisation, carry/future
   decomposition, paired windows, symmetric incremental tests, and conditional-null recursive
   bootstrap inference.
3. **Zero-coupon robustness:** the frozen local GSW/FEDS file supplies genuine zero-coupon yields
   and annual 1–5Y forwards. No network request and no GS fallback are used.

GS results are labelled **CMT-based Fama–Bliss-style**. “Zero-coupon Fama–Bliss” and
“canonical CP” are reserved for the local GSW branch. The model object remains a
**model-implied term-premium signal / macro-anchored excess-yield component**, not a claim that
the residual is a pure structural term premium.


In [29]:
from google.colab import files

uploaded = files.upload()
print("Caricati:", list(uploaded))

Saving feds200628.csv to feds200628.csv
Saving paper_config_monthly_panel.csv to paper_config_monthly_panel (1).csv
Caricati: ['feds200628.csv', 'paper_config_monthly_panel (1).csv']


In [30]:
# ============================================================================
#  LIBRARY CELL - everything inlined, no local imports needed. Collapse me.
#  Hard requirements: numpy, pandas. Optional: scipy (only for the exact CIR-ML
#  refit, NOT needed by any robustness check), pandas_datareader (FRED fallback
#  to plain CSV works without it).
# ============================================================================
import warnings as _w
_w.filterwarnings('ignore', message='Mean of empty slice')
_w.filterwarnings('ignore', message='All-NaN')
import warnings, io, urllib.request, hashlib, json, shutil, platform
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd

# ----------------------------------------------------------------------------
# Constants (identical to the notebooks)
# ----------------------------------------------------------------------------
START = "1961-01-01"
PI_HL = 36            # trend-inflation EWMA half-life (months)
RSTAR_HL = 120        # r* proxy EWMA half-life (months)
MINW = 120            # expanding-window burn-in (months)
H = 12                # forecast horizon (months)
MATS_L = [1, 2, 3, 5, 7, 10]   # paper yield-curve grid (years)
TENORS = [2, 3, 5, 7, 10]      # paper excess-return tenors

# Canonical paper configuration: local frozen inputs, fixed ex ante.
# The CMT snapshot is the canonical July-2026 vintage confirmed for the paper.
PAPER_CURVE_MODE = "gs_cmt_snapshot"
PAPER_RSTAR_MODE = "proxy"
PAPER_EXTERNAL_TP_ID = "THREEFYTP10"
PAPER_ANCHOR_INIT = "post_ewma_bfill"
ALLOW_ECONOMIC_FALLBACKS = False
PAPER_END = pd.Timestamp("2026-05-31")

# Required local inputs. The notebook searches the working directory and /content
# recursively, and refuses files whose byte-level SHA-256 differs.
PAPER_SNAPSHOT_FILENAME = "paper_config_monthly_panel.csv"
PAPER_SNAPSHOT_SHA256 = "67ed3093d17cf5ed849a8cfa3fa95ac5fb2c1f47d58a69110f96a366a1fbd7f5"
PAPER_SNAPSHOT_PANDAS_SHA256 = "c1e32588eb23f57acec28d315973c633852046f68ef6e785cc71657d46989a32"
GSW_LOCAL_FILENAME = "feds200628.csv"
GSW_LOCAL_SHA256 = "c3541ab54e054c0bcc3c17bbe038a3129da2c26486ef3c255e2378d7d63aa9c4"

# Reproducibility and heavier final checks.
WRITE_DATA_SNAPSHOT = True
SNAPSHOT_DIR = Path("macroanchor_data_snapshot")
NOTEBOOK_FILENAME = "08_macroanchor-bond-risk-premia.ipynb"
RUN_ZERO_COUPON_ROBUSTNESS = True
RUN_CONDITIONAL_BOOTSTRAP = True
BOOTSTRAP_B = 9999
BOOTSTRAP_BATCH_SIZE = 250
BOOTSTRAP_BLOCK_LENGTHS = (12, 24)
BOOTSTRAP_SEED = 20260711


# ============================================================================
# 1. DATA LAYER  (from 01/02/03 cells [2]-[3])
# ============================================================================
def fred_series(ids, start=START, end=None):
    """FRED via pandas_datareader, with direct fredgraph CSV fallback."""
    end = end or str(pd.Timestamp.today().date())
    ids = ids if isinstance(ids, list) else [ids]
    try:
        import pandas_datareader.data as web
        df = web.DataReader(ids, "fred", start, end)
        if df is not None and len(df.dropna(how="all")):
            return df
    except Exception as e:
        warnings.warn(f"pandas_datareader failed ({e}); using fredgraph CSV")
    out = {}
    for sid in ids:
        try:
            u = ("https://fred.stlouisfed.org/graph/fredgraph.csv"
                 f"?id={sid}&cosd={pd.Timestamp(start).date()}&coed={pd.Timestamp(end).date()}")
            d = pd.read_csv(u); d.columns = ["date", "val"]
            d["date"] = pd.to_datetime(d["date"])
            s = pd.to_numeric(d.set_index("date")["val"], errors="coerce")
            out[sid] = s.loc[(s.index >= pd.Timestamp(start)) & (s.index <= pd.Timestamp(end))]
        except Exception as e:
            warnings.warn(f"FRED CSV failed for {sid}: {e}")
    return pd.DataFrame(out) if out else None

def _sha256_file(path, chunk_size=1024 * 1024):
    """Byte-level SHA-256 for an immutable input or generated file."""
    path = Path(path); h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


def _resolve_local_file(filename, expected_sha256=None):
    """
    Locate a required local file without network fallback. Searches the current
    directory and /content (including extracted subdirectories).
    """
    filename = str(filename)
    roots = [Path.cwd()]
    content = Path("/content")
    if content.exists() and content.resolve() != Path.cwd().resolve():
        roots.append(content)
    candidates = []
    for root in roots:
        direct = root / filename
        if direct.exists():
            candidates.append(direct)
        try:
            candidates.extend(root.rglob(filename))
        except Exception:
            pass
    unique = []
    seen = set()
    for p in candidates:
        try:
            key = str(p.resolve())
        except Exception:
            key = str(p)
        if key not in seen and p.is_file():
            seen.add(key); unique.append(p)
    if not unique:
        raise FileNotFoundError(
            f"Required local file '{filename}' not found. Upload it to the Colab "
            "session or place it beside the notebook."
        )
    # Prefer the shortest path (normally /content/<filename>).
    path = sorted(unique, key=lambda p: (len(p.parts), str(p)))[0]
    digest = _sha256_file(path)
    if expected_sha256 and digest.lower() != expected_sha256.lower():
        raise RuntimeError(
            f"SHA-256 mismatch for {path}. Expected {expected_sha256}, got {digest}. "
            "Do not resave the file in Excel; use the frozen original."
        )
    return path, digest


def load_paper_snapshot():
    """Load and validate the frozen canonical CMT panel."""
    path, digest = _resolve_local_file(PAPER_SNAPSHOT_FILENAME, PAPER_SNAPSHOT_SHA256)
    df = pd.read_csv(path)
    dcol = next((c for c in df.columns if str(c).strip().lower() == "date"), df.columns[0])
    df[dcol] = pd.to_datetime(df[dcol], errors="raise")
    df = df.set_index(dcol).sort_index()
    df.index.name = "date"
    df = df.loc[df.index <= PAPER_END]
    req = [f"y{m}" for m in MATS_L] + ["TB3MS", "CPIAUCSL", PAPER_EXTERNAL_TP_ID]
    missing = [c for c in req if c not in df.columns]
    if missing:
        raise RuntimeError(f"Canonical snapshot columns missing: {missing}")
    if len(df) != 599 or df.index.min() != pd.Timestamp("1976-06-30") or df.index.max() != PAPER_END:
        raise RuntimeError(
            "Canonical snapshot range mismatch: expected 599 rows, "
            "1976-06-30 to 2026-05-31."
        )
    logical_hash = _hash_frame(df)
    if logical_hash != PAPER_SNAPSHOT_PANDAS_SHA256:
        warnings.warn(
            "Parsed DataFrame hash differs from the archived pandas hash. "
            "The byte-level file hash is valid; this can reflect pandas parsing/version details."
        )
    return df, path, digest, logical_hash


def fetch_gsw(maturities=None):
    """
    Load the frozen local GSW/FEDS zero-coupon file, validate its byte hash,
    convert daily continuously compounded zero yields to month-end decimal units,
    and cut the sample at the canonical paper end date.
    """
    maturities = sorted(set(MATS_L if maturities is None else maturities))
    path, digest = _resolve_local_file(GSW_LOCAL_FILENAME, GSW_LOCAL_SHA256)
    raw = path.read_text(encoding="utf-8", errors="replace")
    lines = raw.splitlines()
    try:
        hdr = next(i for i, line in enumerate(lines) if line.lower().startswith("date,"))
    except StopIteration as exc:
        raise RuntimeError("Could not locate the GSW Date header") from exc
    df = pd.read_csv(io.StringIO("\n".join(lines[hdr:])), na_values=["NA", -999.99, "-999.99"])
    df.columns = [str(c).strip() for c in df.columns]
    dc = df.columns[0]
    df[dc] = pd.to_datetime(df[dc], errors="coerce")
    df = df.dropna(subset=[dc]).set_index(dc).sort_index()
    need = [f"SVENY{m:02d}" for m in maturities]
    missing = [c for c in need if c not in df.columns]
    if missing:
        raise RuntimeError(f"GSW columns missing: {missing}")
    out = df[need].apply(pd.to_numeric, errors="coerce").resample("ME").last() / 100.0
    out = out.loc[out.index <= PAPER_END]
    out.columns = maturities
    out.attrs["source_path"] = str(path)
    out.attrs["sha256_file"] = digest
    return out

def fetch_tp(series_id=PAPER_EXTERNAL_TP_ID):
    """Fetch one pre-specified external 10Y term-premium series; never substitute silently."""
    if series_id is None:
        return None, None
    d = fred_series([series_id])
    if d is None or series_id not in d or d[series_id].notna().sum() <= 50:
        raise RuntimeError(f"Required external TP series unavailable: {series_id}")
    s = d[series_id].resample("ME").last() / 100.0
    s.name = series_id
    return s, series_id

def fetch_hlw():
    """Best-effort HLW r* (NY Fed CSV). -> (Series decimal, src) or (None, None)."""
    urls = ["https://www.newyorkfed.org/medialibrary/media/research/policy/rstar/"
            "Holston_Laubach_Williams_current_estimates.csv"]
    for u in urls:
        try:
            df = pd.read_csv(u)
            dcol = [c for c in df.columns if "date" in c.lower()][0]
            rcands = [c for c in df.columns if "rstar" in c.lower()]
            rcol = ([c for c in rcands if "us" in c.lower()] or rcands)[0]
            s = pd.Series(pd.to_numeric(df[rcol], errors="coerce").values,
                          index=pd.to_datetime(df[dcol])).dropna()
            s = (s / 100.0 if s.median() > 1 else s).resample("ME").last().interpolate()
            if len(s) > 40:
                return s, "HLW (NY Fed)"
        except Exception:
            pass
    return None, None


def load_us_data(curve_mode=PAPER_CURVE_MODE, start=START,
                 external_tp_id=PAPER_EXTERNAL_TP_ID,
                 rstar_mode=PAPER_RSTAR_MODE,
                 allow_fallbacks=ALLOW_ECONOMIC_FALLBACKS):
    """
    Load a fixed local economic specification.

    `gs_cmt_snapshot` reads the frozen canonical CMT panel. `gsw_zero` replaces
    only the yield curve with the frozen local GSW zero-coupon curve while using
    the same TB3MS, CPI and Kim-Wright vintage. No economic fallback is permitted.
    """
    if allow_fallbacks:
        raise ValueError("Economic fallbacks are disabled for the publication notebook")
    curve_mode = str(curve_mode).lower(); rstar_mode = str(rstar_mode).lower()
    snap, snap_path, snap_digest, snap_logical_hash = load_paper_snapshot()

    short_s = pd.to_numeric(snap["TB3MS"], errors="coerce")
    cpi_s = pd.to_numeric(snap["CPIAUCSL"], errors="coerce")
    tp_series = tp_name = None
    if external_tp_id is not None:
        if external_tp_id not in snap.columns:
            raise RuntimeError(f"External TP {external_tp_id} absent from canonical snapshot")
        tp_series = pd.to_numeric(snap[external_tp_id], errors="coerce")
        tp_series.name = external_tp_id; tp_name = external_tp_id

    cp_curve = None; gsw_path = gsw_digest = None
    if curve_mode in ("gs_cmt_snapshot", "gs_cmt"):
        curve = snap[[f"y{m}" for m in MATS_L]].copy()
        curve.columns = MATS_L
        curve_src = "Frozen FRED GS constant-maturity snapshot (par-equivalent)"
        curve_kind = "cmt_par_equivalent"
    elif curve_mode == "gsw_zero":
        full = fetch_gsw(sorted(set(MATS_L + [4])))
        gsw_path = full.attrs.get("source_path")
        gsw_digest = full.attrs.get("sha256_file")
        curve = full[MATS_L].copy()
        cp_curve = full[[1, 2, 3, 4, 5]].copy()
        curve_src = "Frozen Fed GSW zero-coupon (feds200628)"
        curve_kind = "zero_coupon"
    else:
        raise ValueError("curve_mode must be 'gs_cmt_snapshot' or 'gsw_zero'")

    if rstar_mode == "proxy":
        hlw = hlw_src = None
    elif rstar_mode == "hlw":
        # Retained for development only; not used by the canonical notebook.
        hlw, hlw_src = fetch_hlw()
        if hlw is None:
            raise RuntimeError("HLW requested but unavailable")
    else:
        raise ValueError("rstar_mode must be 'proxy' or 'hlw'")

    idx = snap.index.intersection(curve.dropna().index)
    idx = idx[(idx >= pd.Timestamp("1976-06-30")) & (idx <= PAPER_END)]
    dates = pd.DatetimeIndex(idx)
    Ydf = curve.reindex(dates).interpolate(limit_area="inside")
    short_al = short_s.reindex(dates).interpolate(limit_area="inside")
    cpi_al = cpi_s.reindex(dates).interpolate(limit_area="inside")
    if Ydf.isna().any().any() or short_al.isna().any() or cpi_al.isna().any():
        raise RuntimeError("Unresolved missing values in the fixed panel")
    if tp_series is not None:
        tp_series = tp_series.reindex(dates)
    if hlw is not None:
        hlw = hlw.reindex(dates)
    if cp_curve is not None:
        cp_curve = cp_curve.reindex(dates).interpolate(limit_area="inside")
        if cp_curve.isna().any().any():
            raise RuntimeError("Unresolved missing values in annual 1-5Y GSW curve")

    raw_panel = Ydf.copy(); raw_panel.columns = [f"y{m}" for m in MATS_L]
    raw_panel["TB3MS"] = short_al; raw_panel["CPIAUCSL"] = cpi_al
    if tp_series is not None:
        raw_panel[tp_name] = tp_series

    return dict(
        dates=dates, Y=Ydf.to_numpy(), short=short_al.to_numpy(),
        cpi_idx=cpi_al.to_numpy(), tp_series=tp_series, tp_name=tp_name,
        hlw=hlw, hlw_src=hlw_src, curve_src=curve_src,
        curve_kind=curve_kind, curve_mode=curve_mode,
        cp_curve=None if cp_curve is None else cp_curve.to_numpy(),
        cp_taus=None if cp_curve is None else np.array([1, 2, 3, 4, 5], float),
        raw_panel=raw_panel,
        canonical_input_file=str(snap_path),
        canonical_input_sha256_file=snap_digest,
        canonical_input_sha256_pandas=snap_logical_hash,
        gsw_input_file=gsw_path,
        gsw_input_sha256_file=gsw_digest,
    )

# ============================================================================
# 2. THE MACRO ANCHOR  theta_t = r*_t + pi_e,t   (from cell [4])
# ============================================================================
def build_anchor(dates, short, cpi_idx, hlw=None, pi_hl=PI_HL,
                 rstar_hl=RSTAR_HL, initialization=PAPER_ANCHOR_INIT):
    """
    theta_t = r*_t + pi_e,t.

    `post_ewma_bfill` is the new canonical specification: both EWMAs are computed
    from available history first, and only their final initial gaps are backfilled.
    `strict` leaves initial values missing. `draft_pre_rstar_bfill` reproduces the
    old draft ordering in which trend inflation was backfilled before entering r*.
    """
    cpi_yoy = pd.Series(cpi_idx, index=dates).pct_change(12)
    pi_raw = cpi_yoy.ewm(halflife=pi_hl).mean()
    short_s = pd.Series(short, index=dates)

    if hlw is not None:
        pi_trend = pi_raw
        rstar = pd.Series(np.asarray(hlw), index=dates).reindex(dates).interpolate(limit_area="inside").ffill()
        rstar_src = "HLW (NY Fed)"
        if initialization in ("post_ewma_bfill", "draft_pre_rstar_bfill"):
            pi_trend = pi_trend.bfill(); rstar = rstar.bfill()
        elif initialization != "strict":
            raise ValueError("Unknown anchor initialization")
    else:
        if initialization == "draft_pre_rstar_bfill":
            pi_trend = pi_raw.bfill()
            rstar = (short_s - pi_trend).ewm(halflife=rstar_hl).mean().bfill()
        else:
            pi_trend = pi_raw
            rstar = (short_s - pi_trend).ewm(halflife=rstar_hl).mean()
            if initialization == "post_ewma_bfill":
                pi_trend = pi_trend.bfill(); rstar = rstar.bfill()
            elif initialization != "strict":
                raise ValueError(
                    "initialization must be 'post_ewma_bfill', 'strict', "
                    "or 'draft_pre_rstar_bfill'"
                )
        rstar_src = f"proxy: EWMA real short (hl={rstar_hl}m)"
    anchor_sr = (rstar + pi_trend).to_numpy()
    return anchor_sr, rstar.to_numpy(), pi_trend.to_numpy(), rstar_src

# ============================================================================
# 3. PHYSICAL-MEASURE ESTIMATION  (Section A, cell [5])
# ============================================================================
DTm = 1 / 12


def fit_ou(rw):
    """OU/Vasicek by exact AR(1) discretisation, with finite-pair masking."""
    rw = np.asarray(rw, float)
    m = np.isfinite(rw[:-1]) & np.isfinite(rw[1:])
    x = rw[:-1][m]; yv = rw[1:][m]
    if len(x) < 24: return np.nan, np.nan, np.nan
    X = np.column_stack([np.ones(len(x)), x]); c, b = np.linalg.lstsq(X, yv, rcond=None)[0]
    b = float(np.clip(b, 1e-6, 0.999999)); k = -12 * np.log(b); th = float(c / (1 - b))
    e = yv - (c + b * x); s2 = float(np.var(e)) * 2 * k / max(1 - b * b, 1e-10)
    return k, th, np.sqrt(max(s2, 1e-10))

def fit_cir(rw):
    """CIR weighted least squares (QML moments), with finite-pair masking."""
    rw = np.asarray(rw, float)
    m = np.isfinite(rw[:-1]) & np.isfinite(rw[1:])
    x0 = rw[:-1][m]; x1 = rw[1:][m]
    if len(x0) < 24: return np.nan, np.nan, np.nan
    x = np.maximum(x0, 1e-4); d = x1 - x0; w = 1 / np.sqrt(x)
    X = np.column_stack([np.ones(len(x)) * w, x * w]); yv = d * w
    b0, b1 = np.linalg.lstsq(X, yv, rcond=None)[0]
    k = float(np.clip(-b1 * 12, 1e-4, 20)); th = float(np.clip(b0 * 12 / k, 1e-4, 0.5))
    e = d - (b0 + b1 * x); s2 = float(np.mean(e * e / (x * DTm)))
    return k, th, np.sqrt(max(s2, 1e-10))

def cir_ml_refit(r, k0, th0, s0):
    """Exact non-central chi-square ML refinement; returns input if unavailable."""
    if not np.all(np.isfinite([k0, th0, s0])):
        return k0, th0, s0
    try:
        from scipy.stats import ncx2
        from scipy.optimize import minimize
        r = np.asarray(r, float); m = np.isfinite(r[:-1]) & np.isfinite(r[1:])
        r0, r1 = r[:-1][m], r[1:][m]
        def nll(p):
            k, th, s = p
            if k <= 1e-4 or th <= 1e-4 or s <= 1e-4: return 1e9
            c = 2 * k / (s * s * (1 - np.exp(-k * DTm))); df = 4 * k * th / (s * s)
            nc = 2 * c * np.maximum(r0, 1e-6) * np.exp(-k * DTm)
            ll = ncx2.logpdf(2 * c * np.maximum(r1, 1e-6), df, nc) + np.log(2 * c)
            return -np.nansum(ll)
        res = minimize(nll, [k0, th0, s0], method="Nelder-Mead",
                       options={"maxiter": 400, "xatol": 1e-5, "fatol": 1e-3})
        if res.success or res.fun < nll([k0, th0, s0]):
            return tuple(float(v) for v in res.x)
    except Exception:
        pass
    return k0, th0, s0

def fit_hw(rw, anc):
    """HW gap regression with finite-pair masking. Returns (kappa, sigma, t-gap)."""
    rw = np.asarray(rw, float); anc = np.asarray(anc, float)
    m = np.isfinite(rw[:-1]) & np.isfinite(rw[1:]) & np.isfinite(anc[:-1])
    g = anc[:-1][m] - rw[:-1][m]; d = rw[1:][m] - rw[:-1][m]
    if len(g) < 24 or np.sum((g-g.mean())**2) <= 0:
        return np.nan, np.nan, np.nan
    X = np.column_stack([np.ones(len(g)), g]); a0, sl = np.linalg.lstsq(X, d, rcond=None)[0]
    e = d - (a0 + sl * g)
    seb = np.sqrt(np.sum(e * e) / max(len(g) - 2, 1) / np.sum((g - g.mean()) ** 2))
    k = float(np.clip(sl * 12, 1e-4, 20)); s = float(np.std(e)) * np.sqrt(12)
    return k, s, float(sl / seb) if seb > 0 else np.nan

def expanding_params(r, anchor_sr, minw=MINW, cir_ml=True):
    """Expanding-window OU/CIR/HW parameters (no look-ahead). Returns a dict of arrays."""
    Tn = len(r)
    out = {k: np.full(Tn, np.nan) for k in
           ["kOU", "thOU", "sOU", "kC", "thC", "sC", "kHW", "sHW"]}
    for t in range(minw, Tn):
        a, b, c_ = fit_ou(r[:t + 1]); out["kOU"][t], out["thOU"][t], out["sOU"][t] = a, b, c_
        a, b, c_ = fit_cir(r[:t + 1])
        if cir_ml and np.all(np.isfinite([a, b, c_])):
            a, b, c_ = cir_ml_refit(r[:t + 1], a, b, c_)
        out["kC"][t], out["thC"][t], out["sC"][t] = a, b, c_
        a, c_, _ = fit_hw(r[:t + 1], anchor_sr[:t + 1]); out["kHW"][t], out["sHW"][t] = a, c_
    return out


# ============================================================================
# 4. PURE-EXPECTATIONS CURVES -> MODEL TERM PREMIA  (Section B, cell [6])
# ============================================================================
def vas_B(tau, k):
    k = max(k, 1e-4); return (1 - np.exp(-k * tau)) / k


def vas_yield(tau, rr, k, th, s):
    k = max(k, 1e-4); B = vas_B(tau, k)
    lnA = (th - s * s / (2 * k * k)) * (B - tau) - s * s * B * B / (4 * k)
    return (-lnA + B * rr) / tau


def cir_yield(tau, rr, k, th, s):
    k = max(k, 1e-4); s = max(s, 1e-4); g = np.sqrt(k * k + 2 * s * s)
    eg = np.exp(g * tau); den = (g + k) * (eg - 1) + 2 * g
    B = 2 * (eg - 1) / den; lnA = (2 * k * th / (s * s)) * np.log(2 * g * np.exp((k + g) * tau / 2) / den)
    return (-lnA + B * rr) / tau


def model_term_premia(Y, r, anchor_sr, params, taus=None, minw=MINW):
    """Per-tenor model TP = y_obs - y_PE for HW / OU / CIR. Returns (TPHW, TPOU, TPCIR)."""
    taus = np.array(MATS_L, float) if taus is None else np.asarray(taus, float)
    Tn, nta = len(r), len(taus)
    TPHW = np.full((Tn, nta), np.nan); TPOU = np.full((Tn, nta), np.nan); TPCIR = np.full((Tn, nta), np.nan)
    for t in range(minw, Tn):
        for j, tau in enumerate(taus):
            if np.all(np.isfinite([Y[t, j], r[t], params["kOU"][t], params["thOU"][t], params["sOU"][t]])):
                TPOU[t, j] = Y[t, j] - vas_yield(tau, r[t], params["kOU"][t], params["thOU"][t], params["sOU"][t])
            if np.all(np.isfinite([Y[t, j], r[t], params["kHW"][t], anchor_sr[t], params["sHW"][t]])):
                TPHW[t, j] = Y[t, j] - vas_yield(tau, r[t], params["kHW"][t], anchor_sr[t], params["sHW"][t])
            if np.all(np.isfinite([Y[t, j], r[t], params["kC"][t], params["thC"][t], params["sC"][t]])):
                TPCIR[t, j] = Y[t, j] - cir_yield(tau, r[t], params["kC"][t], params["thC"][t], params["sC"][t])
    return TPHW, TPOU, TPCIR


# ============================================================================
# 5. RETURNS AND PREDICTORS  (Section C, cell [8])
# ============================================================================
def y_at(Y, taus, m):
    return np.array([np.interp(m, taus, Y[t]) for t in range(len(Y))])


def excess_returns_fb(Y, taus=None, tenors=None, H=H):
    """
    Zero-coupon Fama-Bliss holding-period returns when Y is a zero-coupon curve.
    With GS constant-maturity/par-equivalent inputs this is a CMT-based FB-style
    approximation and must be labelled as such.
    """
    taus = np.array(MATS_L, float) if taus is None else np.asarray(taus, float)
    tenors = TENORS if tenors is None else list(tenors)
    Tn = len(Y); y1 = y_at(Y, taus, 1); RX = {}
    for n in tenors:
        yn = y_at(Y, taus, n); ynm1 = y_at(Y, taus, n - 1); rx = np.full(Tn, np.nan)
        for t in range(Tn - H):
            if np.all(np.isfinite([yn[t], ynm1[t + H], y1[t]])):
                rx[t] = n * yn[t] - (n - 1) * ynm1[t + H] - y1[t]
        RX[n] = rx
    rx_bar = np.nanmean(np.column_stack([RX[n] for n in tenors]), axis=1)
    return RX, rx_bar

def monthly_returns(Y, r, taus=None):
    """Per-tenor 1m carry+roll+duration+convexity returns; basket excess-of-cash."""
    taus = np.array(MATS_L, float) if taus is None else np.asarray(taus, float)
    Tn, nta = len(Y), len(taus); Dm = taus; Cm = taus ** 2; hh = 1 / 12
    dYm = np.vstack([np.zeros((1, nta)), np.diff(Y, axis=0)])
    rollm = np.zeros_like(Y)
    for t in range(Tn):
        rollm[t] = Dm * (Y[t] - np.interp(taus - hh, taus, Y[t]))
    RET_pt = Y / 12 + rollm - Dm[None, :] * dYm + 0.5 * Cm[None, :] * dYm ** 2
    basket1m = RET_pt[:, 1:].mean(1) - r / 12
    return RET_pt, basket1m


def fama_bliss_fwd(Y, taus=None):
    """Average forward spread (the notebook's fb_bar)."""
    taus = np.array(MATS_L, float) if taus is None else np.asarray(taus, float)
    y1 = Y[:, 0]; fs = []
    for j in range(1, len(taus)):
        f = (taus[j] * Y[:, j] - taus[j - 1] * Y[:, j - 1]) / (taus[j] - taus[j - 1])
        fs.append(f - y1)
    return np.column_stack(fs).mean(1)


def forward_vector(Y, taus=None):
    """
    Forward-rate regressor matrix. On the irregular paper grid this produces a
    CP-style factor (some intervals are multi-year averages). With annual 1..5Y
    zero-coupon inputs it is the canonical one-year-forward construction.
    """
    taus = np.array(MATS_L, float) if taus is None else np.asarray(taus, float)
    cols = [y_at(Y, taus, 1)]
    for j in range(1, len(taus)):
        f = (taus[j] * Y[:, j] - taus[j - 1] * Y[:, j - 1]) / (taus[j] - taus[j - 1])
        cols.append(f)
    return np.column_stack(cols)

def legacy_gap(Y, anchor_sr, tp_add=0.005):
    """The failed raw macro-anchor gap: y10 - (anchor + fixed TP block)."""
    return Y[:, -1] - (anchor_sr + tp_add)


# ============================================================================
# 6. INFERENCE  (Section C + Clark-West cells [8],[14],[16]) -- unified
# ============================================================================
def ols_nw(yv, Xc, L):
    """OLS with Newey-West SE. Xc is a list of regressor columns. Returns (b, se, R2)."""
    X = np.column_stack([np.ones(len(yv))] + [c for c in Xc]); b = np.linalg.lstsq(X, yv, rcond=None)[0]
    e = yv - X @ b; XtX = np.linalg.inv(X.T @ X); Xe = X * e[:, None]; S = Xe.T @ Xe
    for l in range(1, L + 1):
        w = 1 - l / (L + 1); G = Xe[l:].T @ Xe[:-l]; S = S + w * (G + G.T)
    V = XtX @ S @ XtX
    return b, np.sqrt(np.diag(V)), 1 - (e @ e) / (((yv - yv.mean()) ** 2).sum())


def is_r2(P, yv):
    f = (np.isfinite(yv) & np.isfinite(P).all(1)) if P.ndim > 1 else (np.isfinite(yv) & np.isfinite(P))
    Pm = P[f].reshape(-1, 1) if P.ndim == 1 else P[f]; X = np.column_stack([np.ones(f.sum()), Pm])
    b = np.linalg.lstsq(X, yv[f], rcond=None)[0]; e = yv[f] - X @ b
    return 1 - (e @ e) / (((yv[f] - yv[f].mean()) ** 2).sum())


def first_valid_start(P, yv, H=H, minobs=MINW):
    """First origin t where an expanding fit is possible (used to define the common window)."""
    P = np.asarray(P, float); P2 = P.reshape(-1, 1) if P.ndim == 1 else P; yv = np.asarray(yv, float)
    for t in range(minobs + H, len(yv)):
        if not np.isfinite(yv[t]) or not np.isfinite(P2[t]).all():
            continue
        s_ = np.arange(0, t - H + 1); m = np.isfinite(yv[s_]) & np.isfinite(P2[s_]).all(1)
        if m.sum() >= minobs:
            return t
    return None


def oos_r2(P, yv, H=H, minobs=MINW, start=None):
    """
    Campbell-Thompson OOS R2 vs the expanding mean. Unifies oos_r2 / oos_r2_w:
    start=None -> native (first valid origin); start=t0 -> forced common window.
    Returns (R2, n_forecasts).
    """
    P = np.asarray(P, float); P = P.reshape(-1, 1) if P.ndim == 1 else P; yv = np.asarray(yv, float)
    t_start = (minobs + H) if start is None else start
    num = den = 0.0; nf = 0
    for t in range(t_start, len(yv)):
        if not np.isfinite(yv[t]) or not np.isfinite(P[t]).all():
            continue
        s_ = np.arange(0, t - H + 1); m = np.isfinite(yv[s_]) & np.isfinite(P[s_]).all(1)
        if m.sum() < minobs:
            continue
        Xtr = np.column_stack([np.ones(m.sum()), P[s_][m]]); btr = np.linalg.lstsq(Xtr, yv[s_][m], rcond=None)[0]
        num += (yv[t] - (btr[0] + P[t] @ btr[1:])) ** 2; den += (yv[t] - yv[s_][m].mean()) ** 2; nf += 1
    return (1 - num / den if den > 0 else np.nan), nf


def clark_west(P, yv, H=H, minobs=MINW, nwlag=12, start=None):
    """
    Clark-West (2007) adjusted-MSPE test vs the expanding mean (nested).
    One-sided; NW(nwlag). start unifies clark_west / cw_w. Returns (adjMSPE, t, p, n).
    """
    from statistics import NormalDist
    P = np.asarray(P, float); P = P.reshape(-1, 1) if P.ndim == 1 else P; yv = np.asarray(yv, float)
    t_start = (minobs + H) if start is None else start
    fts = []
    for t in range(t_start, len(yv)):
        if not np.isfinite(yv[t]) or not np.isfinite(P[t]).all():
            continue
        s_ = np.arange(0, t - H + 1); m = np.isfinite(yv[s_]) & np.isfinite(P[s_]).all(1)
        if m.sum() < minobs:
            continue
        yb = yv[s_][m].mean()
        Xtr = np.column_stack([np.ones(m.sum()), P[s_][m]]); btr = np.linalg.lstsq(Xtr, yv[s_][m], rcond=None)[0]
        ym = btr[0] + P[t] @ btr[1:]; eb = yv[t] - yb; em = yv[t] - ym
        fts.append(eb * eb - (em * em - (yb - ym) ** 2))
    f = np.array(fts, float); n = len(f)
    if n < 24:
        return np.nan, np.nan, np.nan, n
    u = f - f.mean(); S = np.sum(u * u) / n
    for l in range(1, nwlag + 1):
        w = 1 - l / (nwlag + 1); S += 2 * w * np.sum(u[l:] * u[:-l]) / n
    se = np.sqrt(S / n); tcw = f.mean() / se if se > 0 else np.nan
    p = (1 - NormalDist().cdf(tcw)) if np.isfinite(tcw) else np.nan
    return f.mean(), tcw, p, n


# ============================================================================
# 7. NEW ROBUSTNESS HELPERS  (notebook 08)
# ============================================================================
def orthogonalize_expanding(P, Z, minobs=MINW):
    """
    Strictly past-only expanding residualisation. At origin t, coefficients are
    fitted on s < t; the current observation is projected using those loadings.
    """
    P = np.asarray(P, float); Z = np.asarray(Z, float)
    Z = Z.reshape(-1, 1) if Z.ndim == 1 else Z
    Tn = len(P); out = np.full(Tn, np.nan)
    for t in range(minobs, Tn):
        if not (np.isfinite(P[t]) and np.isfinite(Z[t]).all()): continue
        s_ = np.arange(0, t)
        m = np.isfinite(P[s_]) & np.isfinite(Z[s_]).all(1)
        if m.sum() < minobs: continue
        X = np.column_stack([np.ones(m.sum()), Z[s_][m]])
        b = np.linalg.lstsq(X, P[s_][m], rcond=None)[0]
        out[t] = P[t] - (b[0] + Z[t] @ b[1:])
    return out

def nested_oos_cw(P_small, P_big, yv, H=H, minobs=MINW, nwlag=12, start=None):
    """
    OOS-R2 (big vs small as the benchmark) AND Clark-West for a nested pair of
    predictor SETS. P_small / P_big are (n,) or (n,k) arrays; columns of P_small
    must be a subset of P_big for the CW adjustment to be valid.

    Use to ask "does adding TP to [1, y10] beat [1, y10] out of sample?".
    Returns dict(OOS_R2_big_vs_small, OOS_R2_big_vs_mean, CW_t, CW_p, n).
    """
    from statistics import NormalDist

    def _mat(P):
        P = np.asarray(P, float); return P.reshape(-1, 1) if P.ndim == 1 else P
    Ps, Pb, yv = _mat(P_small), _mat(P_big), np.asarray(yv, float)
    t_start = (minobs + H) if start is None else start
    num_bs = den_bs = num_bm = den_bm = 0.0; fts = []; nf = 0
    for t in range(t_start, len(yv)):
        if not (np.isfinite(yv[t]) and np.isfinite(Ps[t]).all() and np.isfinite(Pb[t]).all()):
            continue
        s_ = np.arange(0, t - H + 1)
        m = np.isfinite(yv[s_]) & np.isfinite(Ps[s_]).all(1) & np.isfinite(Pb[s_]).all(1)
        if m.sum() < minobs:
            continue
        ytr = yv[s_][m]
        Xs = np.column_stack([np.ones(m.sum()), Ps[s_][m]]); bs = np.linalg.lstsq(Xs, ytr, rcond=None)[0]
        Xb = np.column_stack([np.ones(m.sum()), Pb[s_][m]]); bb = np.linalg.lstsq(Xb, ytr, rcond=None)[0]
        f_s = bs[0] + Ps[t] @ bs[1:]; f_b = bb[0] + Pb[t] @ bb[1:]; ybar = ytr.mean()
        e_s = yv[t] - f_s; e_b = yv[t] - f_b; e_m = yv[t] - ybar
        num_bs += e_b ** 2; den_bs += e_s ** 2
        num_bm += e_b ** 2; den_bm += e_m ** 2
        # CW for the (small) benchmark vs (big) alternative
        fts.append(e_s * e_s - (e_b * e_b - (f_s - f_b) ** 2)); nf += 1
    f = np.array(fts, float); n = len(f)
    if n < 24:
        return dict(OOS_R2_big_vs_small=np.nan, OOS_R2_big_vs_mean=np.nan, CW_t=np.nan, CW_p=np.nan, n=n)
    u = f - f.mean(); S = np.sum(u * u) / n
    for l in range(1, nwlag + 1):
        w = 1 - l / (nwlag + 1); S += 2 * w * np.sum(u[l:] * u[:-l]) / n
    se = np.sqrt(S / n); tcw = f.mean() / se if se > 0 else np.nan
    p = (1 - NormalDist().cdf(tcw)) if np.isfinite(tcw) else np.nan
    return dict(OOS_R2_big_vs_small=(1 - num_bs / den_bs) if den_bs > 0 else np.nan,
                OOS_R2_big_vs_mean=(1 - num_bm / den_bm) if den_bm > 0 else np.nan,
                CW_t=tcw, CW_p=p, n=n)


def cp_factor_expanding(Y, rx_bar, taus=None, minobs=MINW, H=H):
    """
    Expanding CP-style fitted forward factor. It is canonical CP only when `Y`
    contains annual 1..5Y zero-coupon yields and rx_bar is the 2..5Y average.
    Forecast loadings use only targets realised by the origin (s+H <= t).
    """
    taus = np.array(MATS_L, float) if taus is None else np.asarray(taus, float)
    F = forward_vector(Y, taus); Tn = len(Y); out = np.full(Tn, np.nan)
    rx_bar = np.asarray(rx_bar, float)
    for t in range(minobs, Tn):
        s_ = np.arange(0, t - H + 1)
        m = np.isfinite(rx_bar[s_]) & np.isfinite(F[s_]).all(1)
        if m.sum() < minobs or not np.isfinite(F[t]).all(): continue
        X = np.column_stack([np.ones(m.sum()), F[s_][m]])
        g = np.linalg.lstsq(X, rx_bar[s_][m], rcond=None)[0]
        out[t] = g[0] + F[t] @ g[1:]
    return out

# ============================================================================
# 8. CONVENIENCE: run the whole US core in one call
# ============================================================================
def build_us_core(curve_mode=PAPER_CURVE_MODE, rstar_mode=PAPER_RSTAR_MODE,
                  external_tp_id=PAPER_EXTERNAL_TP_ID,
                  anchor_init=PAPER_ANCHOR_INIT,
                  allow_fallbacks=ALLOW_ECONOMIC_FALLBACKS,
                  minw=MINW, cir_ml=False, verbose=True):
    """Fixed-configuration assembly: data -> anchor -> model signal -> targets."""
    d = load_us_data(curve_mode=curve_mode, external_tp_id=external_tp_id,
                     rstar_mode=rstar_mode, allow_fallbacks=allow_fallbacks)
    dates, Y, short, cpi_idx = d["dates"], d["Y"], d["short"], d["cpi_idx"]
    r = np.asarray(short, float)
    anchor_sr, rstar, pi_trend, rstar_src = build_anchor(
        dates, short, cpi_idx, hlw=d["hlw"], initialization=anchor_init)
    params = expanding_params(r, anchor_sr, minw=minw, cir_ml=cir_ml)
    taus = np.array(MATS_L, float)
    TPHW, TPOU, TPCIR = model_term_premia(Y, r, anchor_sr, params, taus=taus, minw=minw)
    RX, rx_bar = excess_returns_fb(Y, taus=taus)
    RET_pt, basket1m = monthly_returns(Y, r, taus=taus)
    tpHW_bar = np.where(np.isfinite(TPHW[:, 1:]).all(1), np.mean(TPHW[:, 1:], axis=1), np.nan)
    tpOU_bar = np.where(np.isfinite(TPOU[:, 1:]).all(1), np.mean(TPOU[:, 1:], axis=1), np.nan)
    slope = Y[:, -1] - Y[:, 0]
    fb_bar = fama_bliss_fwd(Y, taus=taus); gap = legacy_gap(Y, anchor_sr)
    target_label = ("zero-coupon Fama-Bliss" if d["curve_kind"] == "zero_coupon"
                    else "CMT-based Fama-Bliss-style")
    if verbose:
        print(f"Assembled {len(dates)} months: {dates[0].date()} -> {dates[-1].date()}")
        print(f"  curve: {d['curve_src']} | r*: {rstar_src} | TP ref: {d['tp_name'] or 'none'}")
        print(f"  target interpretation: {target_label} | anchor init: {anchor_init}")
    return dict(dates=dates, Y=Y, r=r, taus=taus, anchor_sr=anchor_sr,
                rstar=rstar, pi_trend=pi_trend, params=params,
                TPHW=TPHW, TPOU=TPOU, TPCIR=TPCIR, tpHW_bar=tpHW_bar, tpOU_bar=tpOU_bar,
                RX=RX, rx_bar=rx_bar, RET_pt=RET_pt, basket1m=basket1m,
                slope=slope, fb_bar=fb_bar, gap=gap,
                tp_series=d["tp_series"], tp_name=d["tp_name"],
                target_label=target_label, meta=d)


# ============================================================================
# 10. PUBLICATION HELPERS: matched windows, snapshots and finite-sample checks
# ============================================================================
def _as2d(P):
    P = np.asarray(P, float)
    return P.reshape(-1, 1) if P.ndim == 1 else P


def matched_start(*args, yv, H=H, minobs=MINW):
    starts = [first_valid_start(np.asarray(P, float), yv, H=H, minobs=minobs) for P in args]
    starts = [s for s in starts if s is not None]
    return max(starts) if starts else None


def paired_benchmark_table(tp, benchmarks, yv, dates, tp_name="TP_HW", H=H, minobs=MINW):
    """Each benchmark is compared with TP on its own matched window; both incremental directions."""
    rows = []
    tp = np.asarray(tp, float); yv = np.asarray(yv, float)
    for bname, B in benchmarks:
        B = np.asarray(B, float); st = matched_start(tp, B, yv=yv, H=H, minobs=minobs)
        if st is None: continue
        otp, ntp = oos_r2(tp, yv, start=st); _, ctp, ptp, _ = clark_west(tp, yv, start=st)
        ob, nb = oos_r2(B, yv, start=st); _, cb, pb, _ = clark_west(B, yv, start=st)
        add_tp = nested_oos_cw(B, np.column_stack([B, tp]), yv, start=st)
        add_b = nested_oos_cw(tp, np.column_stack([tp, B]), yv, start=st)
        rows.append(dict(benchmark=bname, start=str(pd.DatetimeIndex(dates)[st].date()),
                         n=min(ntp, nb), TP_OOS=otp, TP_CW=ctp,
                         B_OOS=ob, B_CW=cb,
                         TP_adds_R2=add_tp["OOS_R2_big_vs_small"], TP_adds_CW=add_tp["CW_t"],
                         B_adds_R2=add_b["OOS_R2_big_vs_small"], B_adds_CW=add_b["CW_t"]))
    return pd.DataFrame(rows)


def nested_oos_errors(P_small, P_big, yv, H=H, minobs=MINW, start=None):
    """Per-origin squared errors for a nested pair, for regime attribution."""
    Ps, Pb, yv = _as2d(P_small), _as2d(P_big), np.asarray(yv, float)
    t_start = (minobs + H) if start is None else start
    idx=[]; es2=[]; eb2=[]
    for t in range(t_start, len(yv)):
        if not (np.isfinite(yv[t]) and np.isfinite(Ps[t]).all() and np.isfinite(Pb[t]).all()): continue
        s_ = np.arange(0, t-H+1)
        m = np.isfinite(yv[s_]) & np.isfinite(Ps[s_]).all(1) & np.isfinite(Pb[s_]).all(1)
        if m.sum() < minobs: continue
        ys = yv[s_][m]
        Xs=np.column_stack([np.ones(m.sum()), Ps[s_][m]]); bs=np.linalg.lstsq(Xs,ys,rcond=None)[0]
        Xb=np.column_stack([np.ones(m.sum()), Pb[s_][m]]); bb=np.linalg.lstsq(Xb,ys,rcond=None)[0]
        fs=bs[0]+Ps[t]@bs[1:]; fb=bb[0]+Pb[t]@bb[1:]
        idx.append(t); es2.append((yv[t]-fs)**2); eb2.append((yv[t]-fb)**2)
    return np.asarray(idx,int), np.asarray(es2,float), np.asarray(eb2,float)


def _hash_frame(df):
    x = pd.util.hash_pandas_object(df, index=True).values.tobytes()
    return hashlib.sha256(x).hexdigest()


def write_data_snapshot(core, directory=SNAPSHOT_DIR, label="paper_config"):
    """Write the exact in-memory panel and a provenance-rich SHA-256 manifest."""
    directory = Path(directory); directory.mkdir(parents=True, exist_ok=True)
    df = core["meta"]["raw_panel"].copy()
    csv = directory / f"{label}_monthly_panel.csv"
    df.to_csv(csv, index_label="date")
    manifest = {
        "label": label,
        "rows": int(len(df)),
        "start": str(df.index.min().date()),
        "end": str(df.index.max().date()),
        "sha256_pandas": _hash_frame(df),
        "sha256_file": _sha256_file(csv),
        "csv_filename": csv.name,
        "curve_source": core["meta"]["curve_src"],
        "curve_kind": core["meta"]["curve_kind"],
        "rstar_mode": PAPER_RSTAR_MODE,
        "external_tp": core["tp_name"],
        "anchor_initialization": PAPER_ANCHOR_INIT,
        "canonical_input_file": Path(core["meta"]["canonical_input_file"]).name,
        "canonical_input_sha256_file": core["meta"]["canonical_input_sha256_file"],
        "canonical_input_sha256_pandas": core["meta"]["canonical_input_sha256_pandas"],
        "gsw_input_file": (
            Path(core["meta"]["gsw_input_file"]).name
            if core["meta"].get("gsw_input_file") else None
        ),
        "gsw_input_sha256_file": core["meta"].get("gsw_input_sha256_file"),
        "notebook_filename": NOTEBOOK_FILENAME,
        "generated_utc": datetime.now(timezone.utc).isoformat(),
        "python_version": platform.python_version(),
    }
    manifest_path = directory / f"{label}_manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2))
    return csv, manifest

def _circular_block_sample_matrix(x, B, n, block_len, rng):
    """B circular moving-block samples, each of length n."""
    x = np.asarray(x, float); N = len(x)
    n_blocks = int(np.ceil(n / block_len))
    starts = rng.integers(0, N, size=(int(B), n_blocks))
    offsets = np.arange(block_len)
    idx = (starts[:, :, None] + offsets[None, None, :]) % N
    return x[idx].reshape(int(B), -1)[:, :n]


def _recursive_forecast_weight_matrices(P_small, P_big, yv, start,
                                        H=H, minobs=MINW):
    """
    Precompute the exact linear forecast weights for the expanding small and big
    models. This makes 9,999-replication recursive bootstraps practical without
    changing the forecast protocol.
    """
    Ps, Pb, yv = _as2d(P_small), _as2d(P_big), np.asarray(yv, float)
    Tn = len(yv); eval_idx = []; Ws = []; Wb = []
    for t in range(int(start), Tn):
        if not (np.isfinite(yv[t]) and np.isfinite(Ps[t]).all() and np.isfinite(Pb[t]).all()):
            continue
        s_ = np.arange(0, t - H + 1)
        m = np.isfinite(yv[s_]) & np.isfinite(Ps[s_]).all(1) & np.isfinite(Pb[s_]).all(1)
        if m.sum() < minobs:
            continue
        js = s_[m]
        Xs = np.column_stack([np.ones(m.sum()), Ps[js]])
        Xb = np.column_stack([np.ones(m.sum()), Pb[js]])
        xs_t = np.r_[1.0, Ps[t]]
        xb_t = np.r_[1.0, Pb[t]]
        ws = np.zeros(Tn); wb = np.zeros(Tn)
        ws[js] = xs_t @ np.linalg.pinv(Xs)
        wb[js] = xb_t @ np.linalg.pinv(Xb)
        eval_idx.append(t); Ws.append(ws); Wb.append(wb)
    if not eval_idx:
        raise RuntimeError("No valid recursive forecast origins")
    return np.asarray(eval_idx, int), np.asarray(Ws, float), np.asarray(Wb, float)


def _nw_t_rows(f, nwlag=12):
    """Newey-West t statistics row by row for a 2D adjusted-loss matrix."""
    f = np.asarray(f, float)
    if f.ndim == 1:
        f = f[None, :]
    n = f.shape[1]
    mu = np.mean(f, axis=1)
    u = f - mu[:, None]
    S = np.sum(u * u, axis=1) / n
    for lag in range(1, nwlag + 1):
        w = 1 - lag / (nwlag + 1)
        S += 2 * w * np.sum(u[:, lag:] * u[:, :-lag], axis=1) / n
    se = np.sqrt(np.maximum(S, 0.0) / n)
    return np.divide(mu, se, out=np.full_like(mu, np.nan), where=se > 0)


def conditional_null_bootstrap_nested(P_small, P_big, yv, start, B=BOOTSTRAP_B,
                                      block_len=12, seed=BOOTSTRAP_SEED,
                                      H=H, minobs=MINW,
                                      batch_size=BOOTSTRAP_BATCH_SIZE):
    """
    Fast conditional-on-predictors recursive moving-block bootstrap under the
    small-model null. The expanding regressions are exactly re-run through
    precomputed linear forecast weights in every pseudo-sample.
    """
    Ps, Pb, yv = _as2d(P_small), _as2d(P_big), np.asarray(yv, float)
    obs = nested_oos_cw(Ps, Pb, yv, start=start, H=H, minobs=minobs)
    eval_idx, Ws, Wb = _recursive_forecast_weight_matrices(
        Ps, Pb, yv, start=start, H=H, minobs=minobs
    )

    # Internal identity check: precomputed weights must reproduce the observed CW.
    fs_obs = Ws @ np.nan_to_num(yv, nan=0.0)
    fb_obs = Wb @ np.nan_to_num(yv, nan=0.0)
    ye_obs = yv[eval_idx]
    f_obs = (ye_obs - fs_obs) ** 2 - (
        (ye_obs - fb_obs) ** 2 - (fs_obs - fb_obs) ** 2
    )
    cw_fast = _nw_t_rows(f_obs, nwlag=12)[0]
    if not np.isclose(cw_fast, obs["CW_t"], atol=1e-9, rtol=1e-8):
        raise AssertionError(
            f"Fast recursive weights do not reproduce CW: {cw_fast} vs {obs['CW_t']}"
        )

    # Null DGP: full-sample small-model fitted values plus block-resampled residuals.
    m_small = np.isfinite(yv) & np.isfinite(Ps).all(1)
    X = np.column_stack([np.ones(m_small.sum()), Ps[m_small]])
    beta = np.linalg.lstsq(X, yv[m_small], rcond=None)[0]
    fitted_valid = X @ beta
    resid = yv[m_small] - fitted_valid
    resid = resid - resid.mean()
    valid_idx = np.where(m_small)[0]

    rng = np.random.default_rng(seed)
    tstars = []
    remaining = int(B)
    while remaining > 0:
        bnow = min(int(batch_size), remaining)
        rstar = _circular_block_sample_matrix(
            resid, bnow, len(resid), block_len, rng
        )
        ystar = np.zeros((bnow, len(yv)), float)
        ystar[:, valid_idx] = fitted_valid[None, :] + rstar
        fs = ystar @ Ws.T
        fb = ystar @ Wb.T
        ye = ystar[:, eval_idx]
        f = (ye - fs) ** 2 - ((ye - fb) ** 2 - (fs - fb) ** 2)
        tstars.append(_nw_t_rows(f, nwlag=12))
        remaining -= bnow

    tstars = np.concatenate(tstars)
    tstars = tstars[np.isfinite(tstars)]
    p = (1 + np.sum(tstars >= obs["CW_t"])) / (1 + len(tstars)) if len(tstars) else np.nan
    return dict(
        observed_CW=obs["CW_t"],
        bootstrap_p=p,
        B_effective=len(tstars),
        block_len=block_len,
        q90=np.nanquantile(tstars, .90),
        q95=np.nanquantile(tstars, .95),
        q99=np.nanquantile(tstars, .99),
    )

# ============================================================================
# 9. UPGRADED ROBUSTNESS HELPERS (this notebook; incorporate external review)
# ============================================================================
def carry_future_decomposition(Y, taus=None, tenors=None, H=H):
    # Split rx(n)_{t->t+H} into a component KNOWN at t and the FUTURE part:
    #   rx = [n*y_n,t - (n-1)*y_{n-1,t} - y1_t]  +  [-(n-1)*(y_{n-1,t+H} - y_{n-1,t})]
    #      =              carry_t               +             future_t
    # Identity is exact (asserted in the self-test). Predicting carry is trivial
    # (it is known at t); all genuine forecasting content concerns future_t.
    taus = np.array(MATS_L, float) if taus is None else np.asarray(taus, float)
    tenors = TENORS if tenors is None else tenors
    Tn = len(Y); y1 = Y[:, 0]; CARRY = {}; FUT = {}
    for n in tenors:
        ynm1 = y_at(Y, taus, n - 1); jn = taus.tolist().index(n)
        car = np.full(Tn, np.nan); fut = np.full(Tn, np.nan)
        for t in range(Tn - H):
            car[t] = n * Y[t, jn] - (n - 1) * ynm1[t] - y1[t]
            fut[t] = -(n - 1) * (ynm1[t + H] - ynm1[t])
        CARRY[n] = car; FUT[n] = fut
    return CARRY, FUT


def cieslak_povala_expanding(Y, rx_bar, pi_trend, taus=None, minobs=MINW, H=H):
    """
    Strictly real-time Cieslak-Povala-style cycle factor. This is a transparent
    economic benchmark, not a claim of exact replication of their CPI filter.
    """
    tau = np.asarray(pi_trend, float); Tn, K = Y.shape
    C = np.full((Tn, K), np.nan)
    for t in range(minobs, Tn):
        if not np.isfinite(tau[t]): continue
        s_ = np.arange(0, t)  # strictly past-only loadings
        m0 = np.isfinite(tau[s_])
        if m0.sum() < minobs: continue
        for j in range(K):
            m = m0 & np.isfinite(Y[s_, j])
            if m.sum() < minobs: continue
            X = np.column_stack([np.ones(m.sum()), tau[s_][m]])
            b = np.linalg.lstsq(X, Y[s_, j][m], rcond=None)[0]
            C[t, j] = Y[t, j] - (b[0] + b[1] * tau[t])
    c_short = C[:, 0]
    ok_long = np.isfinite(C[:, 1:]).all(1)
    c_long = np.where(ok_long, np.nanmean(C[:, 1:], axis=1), np.nan)
    Z = np.column_stack([c_short, c_long]); rb = np.asarray(rx_bar, float)
    out = np.full(Tn, np.nan)
    for t in range(minobs, Tn):
        s_ = np.arange(0, t - H + 1)
        m = np.isfinite(rb[s_]) & np.isfinite(Z[s_]).all(1)
        if m.sum() < minobs or not np.isfinite(Z[t]).all(): continue
        X = np.column_stack([np.ones(m.sum()), Z[s_][m]])
        g = np.linalg.lstsq(X, rb[s_][m], rcond=None)[0]
        out[t] = g[0] + Z[t] @ g[1:]
    return out

def oos_errors(P, yv, H=H, minobs=MINW, start=None):
    # Same protocol as oos_r2, but returns per-origin squared errors
    # (idx, e_model^2, e_bench^2) so OOS-R2 can be split by subperiod.
    P = np.asarray(P, float); P = P.reshape(-1, 1) if P.ndim == 1 else P
    yv = np.asarray(yv, float); t_start = (minobs + H) if start is None else start
    idx = []; em2 = []; eb2 = []
    for t in range(t_start, len(yv)):
        if not np.isfinite(yv[t]) or not np.isfinite(P[t]).all():
            continue
        s_ = np.arange(0, t - H + 1); m = np.isfinite(yv[s_]) & np.isfinite(P[s_]).all(1)
        if m.sum() < minobs:
            continue
        Xtr = np.column_stack([np.ones(m.sum()), P[s_][m]])
        btr = np.linalg.lstsq(Xtr, yv[s_][m], rcond=None)[0]
        idx.append(t)
        em2.append((yv[t] - (btr[0] + P[t] @ btr[1:])) ** 2)
        eb2.append((yv[t] - yv[s_][m].mean()) ** 2)
    return np.array(idx, int), np.array(em2, float), np.array(eb2, float)


def subperiod_oos_table(dates, idx, em2, eb2, edges):
    # edges: list of (label, lo_date_or_None, hi_date_or_None) with [lo, hi).
    dd = pd.DatetimeIndex(dates)[idx]
    print(f'  {"period":18s} {"n":>4s} {"OOS R2":>9s} {"share of MSPE gap":>19s}')
    total_gap = float(np.sum(eb2 - em2))
    for lab, lo, hi in edges:
        m = np.ones(len(dd), bool)
        if lo is not None: m &= dd >= pd.Timestamp(lo)
        if hi is not None: m &= dd < pd.Timestamp(hi)
        if m.sum() == 0:
            print(f'  {lab:18s} {0:>4d} {"-":>9s} {"-":>19s}'); continue
        r2 = 1 - em2[m].sum() / eb2[m].sum() if eb2[m].sum() > 0 else np.nan
        share = (eb2[m] - em2[m]).sum() / total_gap if total_gap != 0 else np.nan
        print(f'  {lab:18s} {int(m.sum()):>4d} {r2:>+9.4f} {share:>+19.2f}')


def expanding_hw_tp(Y, r, anchor_sr, taus=None, minw=MINW, freeze_at=None):
    # Fast HW-only path: expanding (kappa, sigma) + the TP_HW matrix. If
    # freeze_at=t*, parameters are FROZEN at t* for all t>t* (theta_t still moves:
    # the anchor is an exogenous input) - the 'kappa frozen in 1997' experiment.
    taus_ = np.array(MATS_L, float) if taus is None else np.asarray(taus, float)
    Tn = len(r); kk = np.full(Tn, np.nan); ss = np.full(Tn, np.nan)
    TP = np.full((Tn, len(taus_)), np.nan)
    for t in range(minw, Tn):
        if freeze_at is not None and t > freeze_at:
            k, s = kk[freeze_at], ss[freeze_at]
        else:
            k, s, _ = fit_hw(r[:t + 1], anchor_sr[:t + 1])
        kk[t], ss[t] = k, s
        for j, tau in enumerate(taus_):
            TP[t, j] = Y[t, j] - vas_yield(tau, r[t], k, anchor_sr[t], s)
    return kk, ss, TP


def rolling_hw_kappa(r, anchor_sr, win=120):
    Tn = len(r); out = np.full(Tn, np.nan)
    for t in range(win, Tn):
        k, _, _ = fit_hw(r[t - win:t + 1], anchor_sr[t - win:t + 1]); out[t] = k
    return out


def nonoverlapping_check(P, rx_n, dates, min_obs=15):
    # 12 non-overlapping annual-return series (one per calendar start month):
    # plain OLS t per series (no overlap => no NW needed). Low power, maximal
    # transparency. Returns rows (month, n, beta_on_zscored_P, t).
    P = np.asarray(P, float); yv = np.asarray(rx_n, float)
    months = pd.DatetimeIndex(dates).month; rows = []
    for m0 in range(1, 13):
        sel = np.where((months == m0) & np.isfinite(P) & np.isfinite(yv))[0]
        if len(sel) < min_obs:
            continue
        x = P[sel]; x = (x - x.mean()) / (x.std() if x.std() > 0 else 1.0); yy = yv[sel]
        X = np.column_stack([np.ones(len(x)), x]); b = np.linalg.lstsq(X, yy, rcond=None)[0]
        e = yy - X @ b
        sxx = np.sum((x - x.mean()) ** 2)
        se = np.sqrt((e @ e) / max(len(x) - 2, 1) / sxx) if sxx > 0 else np.nan
        rows.append((m0, len(x), float(b[1]), float(b[1] / se) if se and np.isfinite(se) else np.nan))
    return rows


## Self-test (offline, synthetic)

The test now checks more than “the code ran”:

- exact carry + future = return identity;
- strictly past-only invariance to changes in future observations;
- successful matched-window and symmetric incremental comparisons;
- conditional-null bootstrap path on a small number of replications;
- monotone synthetic tenor structure.

In [31]:
def make_synth(T=540, seed=7):
    rng = np.random.default_rng(seed)
    dates = pd.date_range("1980-01-31", periods=T, freq="ME"); taus = np.array(MATS_L, float)
    rstar = np.clip(0.02 + 0.003*np.cumsum(rng.standard_normal(T))/np.sqrt(T), -0.005, 0.045)
    pim = 0.028 + 0.018*np.sin(np.linspace(0, 2.5*np.pi, T)); pi = np.empty(T); pi[0] = 0.035
    for t in range(1, T): pi[t] = pi[t-1] + 0.12*(pim[t]-pi[t-1]) + 0.003*rng.standard_normal()
    pi = np.clip(pi, 0.002, 0.09); theta = rstar + pi
    r = np.empty(T); r[0] = theta[0]; KAP, SIG = 0.16, 0.008
    for t in range(1, T): r[t] = r[t-1] + KAP/12*(theta[t]-r[t-1]) + SIG/np.sqrt(12)*rng.standard_normal()
    r = np.clip(r, 5e-4, 0.13); cpi = np.empty(T); cpi[0] = 100.0
    for t in range(1, T): cpi[t] = cpi[t-1]*(1 + pi[t]/12 + 8e-4*rng.standard_normal())
    tp = np.empty(T); tp[0] = 0.012
    for t in range(1, T): tp[t] = 0.012 + 0.95*(tp[t-1]-0.012) + 0.0022*rng.standard_normal()
    Y = np.full((T, len(taus)), np.nan)
    for t in range(T):
        for j, tau in enumerate(taus):
            Y[t, j] = vas_yield(tau, r[t], KAP, theta[t], SIG) + tp[t]*(tau/10.0)**0.9 + 4e-4*rng.standard_normal()
    return dates, np.clip(Y, 2e-4, 0.16), r, cpi

d_, Y_, r_, cpi_ = make_synth()
anc_, rs_, pit_, _ = build_anchor(d_, r_, cpi_, hlw=None)
kk_, ss_, TPH_ = expanding_hw_tp(Y_, r_, anc_)
RX_, rxb_ = excess_returns_fb(Y_)
tpb_ = np.where(np.isfinite(TPH_[:, 1:]).all(1), TPH_[:, 1:].mean(1), np.nan)
y10_ = Y_[:, -1]; rx10_ = RX_[10]; tp10_ = TPH_[:, -1]

# 1) exact decomposition identity
CAR_, FUT_ = carry_future_decomposition(Y_)
gap_ = np.nanmax(np.abs(CAR_[10] + FUT_[10] - RX_[10]))
assert gap_ < 1e-10, gap_

# 2) strict past-only invariance: perturbing future observations cannot alter earlier residuals
lev_ = Y_.mean(1); slp_ = Y_[:, -1]-Y_[:, 0]; crv_ = 2*Y_[:, 3]-Y_[:, 0]-Y_[:, -1]
o1_ = orthogonalize_expanding(tp10_, np.column_stack([lev_, slp_, crv_]))
tcut = 330
tp_mut = tp10_.copy(); tp_mut[tcut+1:] += 100.0
o2_ = orthogonalize_expanding(tp_mut, np.column_stack([lev_, slp_, crv_]))
assert np.nanmax(np.abs(o1_[:tcut+1]-o2_[:tcut+1])) < 1e-12

# 3) monotone tenor structure recovered
nts=[]
for j,n in enumerate(TENORS):
    p=TPH_[:,j+1]; yv=RX_[n]; f=np.isfinite(p)&np.isfinite(yv)
    pz=(p[f]-p[f].mean())/p[f].std(); b,se,_=ols_nw(yv[f],[pz],H); nts.append(b[1]/se[1])
assert nts[-1] > nts[0]

# 4) all publication paths execute
cp_=cp_factor_expanding(Y_,rxb_); cpp_=cieslak_povala_expanding(Y_,rxb_,pit_)
t0_=matched_start(tpb_,cp_,yv=rxb_)
tab_=paired_benchmark_table(tpb_,[("CP-style",cp_),("CP-P-style",cpp_)],rxb_,d_)
assert len(tab_)==2 and np.isfinite(tab_["TP_OOS"]).all()
nst_=nested_oos_cw(y10_,np.column_stack([y10_,tp10_]),rx10_,start=matched_start(y10_,tp10_,yv=rx10_))
assert np.isfinite(nst_["CW_t"])
boot_=conditional_null_bootstrap_nested(y10_,np.column_stack([y10_,tp10_]),rx10_,
        start=matched_start(y10_,tp10_,yv=rx10_),B=19,block_len=12,seed=9)
assert boot_["B_effective"] >= 15
print(f"identity {gap_:.2e} | past-only invariant | tenor t {nts[0]:.2f}->{nts[-1]:.2f} | "
      f"matched rows {len(tab_)} | bootstrap reps {boot_['B_effective']}")
print("SELF-TEST OK")

identity 1.11e-16 | past-only invariant | tenor t 0.85->3.66 | matched rows 2 | bootstrap reps 19
SELF-TEST OK



## Load the frozen canonical paper configuration

The default run now reads `paper_config_monthly_panel.csv` locally and validates its archived
SHA-256. It contains the confirmed 599-month vintage from June 1976 through May 2026:

- GS constant-maturity/par-equivalent curve;
- `TB3MS`;
- `CPIAUCSL`;
- Kim–Wright `THREEFYTP10`.

The canonical anchor is `post_ewma_bfill`: trend inflation and the proxy for \(r^*\) are each
estimated from available history before their remaining initial gaps are backfilled. The old draft
ordering is retained only as an explicitly named audit specification.


In [32]:
DATA_OK = False
try:
    core = build_us_core(curve_mode=PAPER_CURVE_MODE, rstar_mode=PAPER_RSTAR_MODE,
                         external_tp_id=PAPER_EXTERNAL_TP_ID,
                         anchor_init=PAPER_ANCHOR_INIT,
                         allow_fallbacks=False, cir_ml=False, verbose=True)
    dates = core["dates"]; Y = core["Y"]; r = core["r"]
    RX = core["RX"]; rx_bar = core["rx_bar"]; rx10 = RX[10]
    tpHW_bar = core["tpHW_bar"]; TPHW = core["TPHW"]; tp10 = TPHW[:, -1]
    slope = core["slope"]; fb_bar = core["fb_bar"]; y10 = Y[:, -1]
    pi_trend = core["pi_trend"]; anchor_sr = core["anchor_sr"]; params = core["params"]
    tp_series, tp_name = core["tp_series"], core["tp_name"]
    cpi_idx = core["meta"]["cpi_idx"]; target_label = core["target_label"]
    t0 = first_valid_start(tpHW_bar, rx_bar)
    print(f"\ncanonical-window start: {dates[t0].date()} | external TP: {tp_name}")
    print(f"target label: {target_label}")
    print("\nBASELINE sanity check, basket, paper window:")
    for nm, P in [("TP_HW", tpHW_bar), ("slope", slope), ("Fama-Bliss-style", fb_bar), ("legacy gap", core["gap"])]:
        o, n = oos_r2(np.asarray(P, float), rx_bar, start=t0)
        _, cw, p, _ = clark_west(np.asarray(P, float), rx_bar, start=t0)
        print(f"  {nm:18s} OOS {o:+.4f}  CW {cw:5.2f}  n {n}")
    if WRITE_DATA_SNAPSHOT:
        snap, manifest = write_data_snapshot(core, label="paper_config_derived")
        print(f"\nsnapshot: {snap}")
        print(f"panel hash: {manifest['sha256_pandas']}")
    DATA_OK = True
except Exception as e:
    print("CANONICAL REAL-DATA CONFIGURATION UNAVAILABLE ->", repr(e))
    print("No economically different fallback was used. Synthetic checks remain valid.")

/tmp/ipykernel_2290/1117197058.py:156: UserWarning: Parsed DataFrame hash differs from the archived pandas hash. The byte-level file hash is valid; this can reflect pandas parsing/version details.
  warnings.warn(


Assembled 599 months: 1976-06-30 -> 2026-05-31
  curve: Frozen FRED GS constant-maturity snapshot (par-equivalent) | r*: proxy: EWMA real short (hl=120m) | TP ref: THREEFYTP10
  target interpretation: CMT-based Fama-Bliss-style | anchor init: post_ewma_bfill

canonical-window start: 1997-05-31 | external TP: THREEFYTP10
target label: CMT-based Fama-Bliss-style

BASELINE sanity check, basket, paper window:
  TP_HW              OOS +0.0163  CW  1.37  n 336
  slope              OOS -0.0100  CW  1.60  n 336
  Fama-Bliss-style   OOS -0.0154  CW  1.62  n 336
  legacy gap         OOS -0.4445  CW  0.77  n 336

snapshot: macroanchor_data_snapshot/paper_config_derived_monthly_panel.csv
panel hash: 9bdd2f4e1f6fd4f5cfb5e08e60c626b8901f5c1045727208932c951b34003a3c


## Check 1 — level/carry identification

For the GS baseline the target is explicitly **CMT-based Fama–Bliss-style**, not exact zero-coupon.

1. Nested Clark–West: $[1,y_{10},TP_{10}]$ versus $[1,y_{10}]$.
2. Strictly past-only residualisation of $TP_{10}$ on $y_{10}$ and on level/slope/curvature.
3. Decomposition into known-at-origin carry and the future price/yield-change component.

A positive nested result establishes predictive content beyond the contemporaneous level in this
specification. It does not by itself identify a pure structural term premium.

In [33]:
if not DATA_OK:
    print("(skipped - canonical real data unavailable)")
else:
    print(f"CHECK 1 — {target_label} 10Y target\n")
    t1 = matched_start(y10, tp10, yv=rx10)
    n1 = nested_oos_cw(y10, np.column_stack([y10, tp10]), rx10, start=t1)
    print("(1a) [1, y10, TP10] vs [1, y10]")
    print(f"     big-vs-small R2 {n1['OOS_R2_big_vs_small']:+.4f} | big-vs-mean {n1['OOS_R2_big_vs_mean']:+.4f}"
          f" | CW {n1['CW_t']:.2f} | p {n1['CW_p']:.4f} | n {n1['n']}")

    lev = Y.mean(1); crv = 2*Y[:, 3] - Y[:, 0] - Y[:, -1]
    tp10_o1 = orthogonalize_expanding(tp10, y10)
    tp10_o3 = orthogonalize_expanding(tp10, np.column_stack([lev, slope, crv]))
    t2 = matched_start(tp10, tp10_o1, tp10_o3, yv=rx10)
    print(f"\n(1b) strictly past-only residualisation; matched start {dates[t2].date()}")
    for nm, P in [("TP10 raw", tp10), ("TP10 | y10", tp10_o1), ("TP10 | L,S,C", tp10_o3)]:
        o, n = oos_r2(np.asarray(P,float), rx10, start=t2)
        _, cw, p, _ = clark_west(np.asarray(P,float), rx10, start=t2)
        print(f"     {nm:14s} OOS {o:+.4f}  CW {cw:5.2f}  p {p:.4f}  n {n}")

    CAR, FUT = carry_future_decomposition(Y)
    chk = np.nanmax(np.abs(CAR[10] + FUT[10] - RX[10]))
    assert chk < 1e-10
    fut10 = FUT[10]; t3 = matched_start(y10, tp10, yv=fut10)
    o, n = oos_r2(tp10, fut10, start=t3); _, cw, p, _ = clark_west(tp10, fut10, start=t3)
    n3 = nested_oos_cw(y10, np.column_stack([y10, tp10]), fut10, start=t3)
    print(f"\n(1c) carry+future identity max error {chk:.2e}")
    print(f"     TP10 -> future component: OOS {o:+.4f}  CW {cw:.2f}  p {p:.4f}  n {n}")
    print(f"     [1,y10,TP10] vs [1,y10] on future: R2 {n3['OOS_R2_big_vs_small']:+.4f}"
          f"  CW {n3['CW_t']:.2f}  p {n3['CW_p']:.4f}  n {n3['n']}")

CHECK 1 — CMT-based Fama-Bliss-style 10Y target

(1a) [1, y10, TP10] vs [1, y10]
     big-vs-small R2 +0.1441 | big-vs-mean +0.0409 | CW 3.15 | p 0.0008 | n 336

(1b) strictly past-only residualisation; matched start 2007-05-31
     TP10 raw       OOS +0.1395  CW  2.14  p 0.0161  n 216
     TP10 | y10     OOS +0.1045  CW  2.24  p 0.0126  n 216
     TP10 | L,S,C   OOS +0.1535  CW  2.49  p 0.0063  n 216

(1c) carry+future identity max error 2.22e-16
     TP10 -> future component: OOS +0.0323  CW 1.99  p 0.0234  n 336
     [1,y10,TP10] vs [1,y10] on future: R2 +0.0602  CW 2.77  p 0.0028  n 336


## Check 2 — paired-window benchmark horse races

The GS baseline supports a **CP-style irregular-grid forward factor**, not a canonical CP replica:
the 3→5, 5→7 and 7→10 regressors are multi-year average forwards and the underlying CMT curve
is par-equivalent. The optional GSW section below constructs the canonical annual 1–5Y
zero-coupon version.

Every benchmark is compared with $TP_{HW}$ on its own matched window. The table reports both
incremental directions, preventing a late-starting series from silently redefining the entire horse
race. Cieslak–Povala remains explicitly “style”; Kim–Wright is explicitly pseudo-OOS.

In [34]:
if not DATA_OK:
    print("(skipped - canonical real data unavailable)")
else:
    cp_style = cp_factor_expanding(Y, rx_bar)
    cpp_style = cieslak_povala_expanding(Y, rx_bar, pi_trend)
    ext = np.asarray(tp_series, float) if tp_series is not None else None

    benches = [("CP-style irregular-grid", cp_style),
               ("Cieslak-Povala-style", cpp_style),
               ("slope", slope), ("Fama-Bliss-style", fb_bar)]
    if ext is not None: benches.append((f"Kim-Wright pseudo-OOS ({tp_name})", ext))

    print("CHECK 2A — basket target, pairwise matched windows")
    basket_table = paired_benchmark_table(tpHW_bar, benches, rx_bar, dates, tp_name="TP_HW")
    print(basket_table.to_string(index=False, float_format=lambda x: f"{x:+.4f}"))

    print("\nCHECK 2B — 10Y target, pairwise matched windows")
    ten_table = paired_benchmark_table(tp10, benches, rx10, dates, tp_name="TP_HW10")
    print(ten_table.to_string(index=False, float_format=lambda x: f"{x:+.4f}"))

    # Explicit external-TP symmetry, highlighted for the referee response.
    if ext is not None:
        st_ext10 = matched_start(tp10, ext, yv=rx10)
        tp_over_ext = nested_oos_cw(ext, np.column_stack([ext,tp10]), rx10, start=st_ext10)
        ext_over_tp = nested_oos_cw(tp10, np.column_stack([tp10,ext]), rx10, start=st_ext10)
        print(f"\nKim-Wright 10Y symmetry, start {dates[st_ext10].date()}:")
        print(f"  TP10 adds over KW: R2 {tp_over_ext['OOS_R2_big_vs_small']:+.4f}  CW {tp_over_ext['CW_t']:.2f}  p {tp_over_ext['CW_p']:.4f}")
        print(f"  KW adds over TP10: R2 {ext_over_tp['OOS_R2_big_vs_small']:+.4f}  CW {ext_over_tp['CW_t']:.2f}  p {ext_over_tp['CW_p']:.4f}")

CHECK 2A — basket target, pairwise matched windows
                          benchmark      start   n  TP_OOS   TP_CW   B_OOS    B_CW  TP_adds_R2  TP_adds_CW  B_adds_R2  B_adds_CW
            CP-style irregular-grid 1998-04-30 325 +0.0216 +1.3996 -0.0291 -0.9435     +0.0466     +1.3824    -0.0346    +0.3189
               Cieslak-Povala-style 2008-04-30 205 +0.1021 +2.1183 +0.0308 +1.3541     +0.0209     +1.2096    -0.0488    -0.0730
                              slope 1997-05-31 336 +0.0163 +1.3659 -0.0100 +1.5971     -0.0778     +0.8138    -0.0777    +0.2713
                   Fama-Bliss-style 1997-05-31 336 +0.0163 +1.3659 -0.0154 +1.6159     -0.0880     +0.6594    -0.0848    +0.1748
Kim-Wright pseudo-OOS (THREEFYTP10) 2000-12-31 293 -0.0285 +0.9149 -0.0415 +1.8427     -0.1004     -1.2194    -0.1375    -0.6276

CHECK 2B — 10Y target, pairwise matched windows
                          benchmark      start   n  TP_OOS   TP_CW   B_OOS    B_CW  TP_adds_R2  TP_adds_CW  B_adds_R2  B_adds_

## Regime diagnostics and overlap-free descriptive evidence

The rolling estimate is evidence about local identification, not proof that the true parameter
changed. The frozen-1997 experiment asks whether modern updating matters. Subperiod tables are
reported for the basket, the 10Y target, the future component, and the nested gain over the level.

The twelve non-overlapping annual regressions are an **overlap-free descriptive in-sample check**;
they do not replace finite-sample OOS inference.

In [35]:
if not DATA_OK:
    print("(skipped - canonical real data unavailable)")
else:
    kHW = params["kHW"]; kroll = rolling_hw_kappa(r, anchor_sr, win=120)
    yrs = [y for y in range(1990, dates[-1].year + 1, 5)]
    print("kappa_HW, expanding vs rolling(120m), December snapshots:")
    for y_ in yrs:
        sel = np.where((pd.DatetimeIndex(dates).year == y_) & (pd.DatetimeIndex(dates).month == 12))[0]
        if len(sel)==0: continue
        t=sel[0]; kr=f"{kroll[t]:.3f}" if np.isfinite(kroll[t]) else "-"; ke=f"{kHW[t]:.3f}" if np.isfinite(kHW[t]) else "-"
        print(f"  {y_}: expanding {ke:>6s}   rolling {kr:>6s}")

    i97 = int(np.where(pd.DatetimeIndex(dates) <= pd.Timestamp("1997-12-31"))[0][-1])
    _, _, TPfz = expanding_hw_tp(Y, r, anchor_sr, freeze_at=i97)
    tpb_fz = np.where(np.isfinite(TPfz[:,1:]).all(1), TPfz[:,1:].mean(1), np.nan)
    print(f"\nfrozen in {dates[i97].date()} (kappa/sigma fixed; anchor still moves):")
    for nm,P in [("expanding",tpHW_bar),("frozen-1997",tpb_fz)]:
        o,n=oos_r2(np.asarray(P,float),rx_bar,start=t0); _,cw,p,_=clark_west(np.asarray(P,float),rx_bar,start=t0)
        print(f"  {nm:12s} OOS {o:+.4f}  CW {cw:.2f}  p {p:.4f}  n {n}")

    edges=[("1997-2007",None,"2008-01-01"),("2008-2015","2008-01-01","2016-01-01"),("2016-end","2016-01-01",None)]
    print("\nBasket TP_HW vs expanding mean:")
    idx,em2,eb2=oos_errors(tpHW_bar,rx_bar,start=t0); subperiod_oos_table(dates,idx,em2,eb2,edges)
    print("\n10Y TP_HW vs expanding mean:")
    st10=first_valid_start(tp10,rx10); idx,em2,eb2=oos_errors(tp10,rx10,start=st10); subperiod_oos_table(dates,idx,em2,eb2,edges)
    CAR,FUT=carry_future_decomposition(Y); fut10=FUT[10]
    print("\n10Y future component, TP_HW vs expanding mean:")
    stf=first_valid_start(tp10,fut10); idx,em2,eb2=oos_errors(tp10,fut10,start=stf); subperiod_oos_table(dates,idx,em2,eb2,edges)
    print("\nNested gain on future component: [y10] small vs [y10,TP10] big:")
    stn=matched_start(y10,tp10,yv=fut10); idx,es2,ebig2=nested_oos_errors(y10,np.column_stack([y10,tp10]),fut10,start=stn)
    subperiod_oos_table(dates,idx,ebig2,es2,edges)

    print("\n12 non-overlapping annual regressions (descriptive, not OOS CW):")
    rows=nonoverlapping_check(tp10,rx10,dates); pos=sum(1 for _,_,b,_ in rows if b>0)
    for m0,n,b,tstat in rows: print(f"  month {m0:>2}: n {n:>3}  beta {b*100:+.2f}%  t {tstat:+.2f}")
    med_t=np.median([tt for *_,tt in rows if np.isfinite(tt)])
    print(f"  positive slopes {pos}/{len(rows)}, median t {med_t:+.2f}")

kappa_HW, expanding vs rolling(120m), December snapshots:
  1990: expanding  0.950   rolling  0.740
  1995: expanding  0.546   rolling  0.000
  2000: expanding  0.507   rolling  0.000
  2005: expanding  0.370   rolling  0.008
  2010: expanding  0.242   rolling  0.124
  2015: expanding  0.221   rolling  0.045
  2020: expanding  0.215   rolling  0.026
  2025: expanding  0.197   rolling  0.263

frozen in 1997-12-31 (kappa/sigma fixed; anchor still moves):
  expanding    OOS +0.0163  CW 1.37  p 0.0860  n 336
  frozen-1997  OOS -0.0202  CW 0.96  p 0.1695  n 336

Basket TP_HW vs expanding mean:
  period                n    OOS R2   share of MSPE gap
  1997-2007           128   -0.0782               -1.85
  2008-2015            96   -0.1595               -1.29
  2016-end            112   +0.1393               +4.14

10Y TP_HW vs expanding mean:
  period                n    OOS R2   share of MSPE gap
  1997-2007           128   -0.0890               -0.30
  2008-2015            96   +0.0686   


## Anchor robustness: half-life surface and initialization audit

The half-life grid is a sensitivity surface, not a search. The canonical initialization is
`post_ewma_bfill`. Two explicit alternatives are reported:

- `strict`: no initial backfill;
- `draft_pre_rstar_bfill`: the old draft ordering, where trend inflation was backfilled before
  entering the \(r^*\) EWMA.

This makes the earlier silent initialization drift observable rather than hiding it behind a
“paper replication” label.


In [36]:

if not DATA_OK:
    print("(skipped - canonical real data unavailable)")
else:
    rows = []
    for hp in [24, 36, 60]:
        for hr in [60, 120, 180]:
            anc2, *_ = build_anchor(
                dates, r, cpi_idx, hlw=None, pi_hl=hp, rstar_hl=hr,
                initialization=PAPER_ANCHOR_INIT
            )
            _, _, TP2 = expanding_hw_tp(Y, r, anc2)
            tpb2 = np.where(np.isfinite(TP2[:, 1:]).all(1), TP2[:, 1:].mean(1), np.nan)
            tg = first_valid_start(tpb2, rx_bar)
            o, n = oos_r2(tpb2, rx_bar, start=tg)
            _, cw, p, _ = clark_west(tpb2, rx_bar, start=tg)
            rows.append(dict(
                pi_hl=hp, rstar_hl=hr, OOS_R2=o, CW=cw, p=p, n=n,
                baseline=(hp == 36 and hr == 120)
            ))
    sens = pd.DataFrame(rows)
    print(f"half-life surface ({PAPER_ANCHOR_INIT}):")
    print(sens.to_string(index=False, float_format=lambda x: f"{x:+.4f}"))

    init_specs = [
        ("canonical post-EWMA bfill", "post_ewma_bfill"),
        ("strict no backfill", "strict"),
        ("old draft pre-r* bfill", "draft_pre_rstar_bfill"),
    ]
    init_signals = []
    init_anchors = []
    for label, mode in init_specs:
        anc, _, _, _ = build_anchor(
            dates, r, cpi_idx, hlw=None, pi_hl=PI_HL,
            rstar_hl=RSTAR_HL, initialization=mode
        )
        _, _, TPx = expanding_hw_tp(Y, r, anc)
        sig = np.where(np.isfinite(TPx[:, 1:]).all(1), TPx[:, 1:].mean(1), np.nan)
        init_signals.append((label, mode, sig))
        init_anchors.append((label, anc))

    st = matched_start(*[x[2] for x in init_signals], yv=rx_bar)
    print(f"\ninitialization audit, matched start {dates[st].date()}:")
    for label, mode, P in init_signals:
        o, n = oos_r2(P, rx_bar, start=st)
        _, cw, p, _ = clark_west(P, rx_bar, start=st)
        print(f"  {label:28s} OOS {o:+.4f}  CW {cw:.2f}  p {p:.4f}  n {n}")

    base_anchor = init_anchors[0][1]
    for label, anc in init_anchors[1:]:
        diff_bp = np.nanmax(np.abs(anc - base_anchor)) * 1e4
        print(f"  max |anchor - canonical|, {label}: {diff_bp:.2f} bp")


half-life surface (post_ewma_bfill):
 pi_hl  rstar_hl  OOS_R2      CW       p   n  baseline
    24        60 +0.0225 +1.1617 +0.1227 336     False
    24       120 +0.0235 +1.1929 +0.1165 336     False
    24       180 +0.0235 +1.2064 +0.1138 336     False
    36        60 +0.0154 +1.2871 +0.0990 336     False
    36       120 +0.0163 +1.3659 +0.0860 336      True
    36       180 +0.0162 +1.3946 +0.0816 336     False
    60        60 +0.0105 +1.4796 +0.0695 336     False
    60       120 +0.0144 +1.6066 +0.0541 336     False
    60       180 +0.0152 +1.6485 +0.0496 336     False

initialization audit, matched start 1997-05-31:
  canonical post-EWMA bfill    OOS +0.0163  CW 1.37  p 0.0860  n 336
  strict no backfill           OOS +0.0169  CW 1.37  p 0.0851  n 336
  old draft pre-r* bfill       OOS +0.0116  CW 1.31  p 0.0943  n 336
  max |anchor - canonical|, strict no backfill: 0.00 bp
  max |anchor - canonical|, old draft pre-r* bfill: 67.36 bp



## Zero-coupon robustness — frozen local GSW/FEDS input

This section reads the validated local `feds200628.csv`, cuts it at May 2026, and re-runs the key
tests with genuine zero-coupon yields. It constructs annual 1–5Y zero-coupon forwards and the
canonical CP factor. The branch makes no HTTP request and never falls back to GS/CMT.


In [37]:

if not RUN_ZERO_COUPON_ROBUSTNESS:
    print("(zero-coupon robustness disabled in configuration)")
else:
    try:
        zc = build_us_core(
            curve_mode="gsw_zero",
            rstar_mode=PAPER_RSTAR_MODE,
            external_tp_id=PAPER_EXTERNAL_TP_ID,
            anchor_init=PAPER_ANCHOR_INIT,
            allow_fallbacks=False,
            cir_ml=False,
            verbose=True,
        )
        zd = zc["dates"]; zY = zc["Y"]; zrx = zc["rx_bar"]; zrx10 = zc["RX"][10]
        ztpb = zc["tpHW_bar"]; ztp10 = zc["TPHW"][:, -1]; zy10 = zY[:, -1]

        print("\nGSW ZERO-COUPON HEADLINE")
        for nm, P, yv in [("basket", ztpb, zrx), ("10Y", ztp10, zrx10)]:
            st = first_valid_start(P, yv)
            o, n = oos_r2(P, yv, start=st)
            _, cw, p, _ = clark_west(P, yv, start=st)
            print(f"  {nm:8s} OOS {o:+.4f}  CW {cw:.2f}  p {p:.4f}  n {n}")

        st = matched_start(zy10, ztp10, yv=zrx10)
        rr = nested_oos_cw(zy10, np.column_stack([zy10, ztp10]), zrx10, start=st)
        print(
            "  10Y TP adds beyond y10: "
            f"R2 {rr['OOS_R2_big_vs_small']:+.4f}  "
            f"CW {rr['CW_t']:.2f}  p {rr['CW_p']:.4f}"
        )

        zcpY = zc["meta"]["cp_curve"]; zcpT = zc["meta"]["cp_taus"]
        _, zrxcpbar = excess_returns_fb(zcpY, taus=zcpT, tenors=[2, 3, 4, 5])
        zcp = cp_factor_expanding(zcpY, zrxcpbar, taus=zcpT)
        print("\nCANONICAL CP — annual 1–5Y zero-coupon, paired comparisons")
        print(
            paired_benchmark_table(
                ztpb, [("canonical CP", zcp)], zrx, zd
            ).to_string(index=False, float_format=lambda x: f"{x:+.4f}")
        )
        print(
            paired_benchmark_table(
                ztp10, [("canonical CP", zcp)], zrx10, zd
            ).to_string(index=False, float_format=lambda x: f"{x:+.4f}")
        )

        # Finite-sample checks for the two zero-coupon identification headlines.
        if RUN_CONDITIONAL_BOOTSTRAP:
            zempty = np.empty((len(zrx), 0))
            ztests = [
                ("ZC basket: mean -> TP_HW", zempty, ztpb, zrx, first_valid_start(ztpb, zrx)),
                ("ZC M1 10Y: y10 -> y10+TP10",
                 zy10, np.column_stack([zy10, ztp10]), zrx10,
                 matched_start(zy10, ztp10, yv=zrx10)),
            ]
            print("\nZERO-COUPON CONDITIONAL BOOTSTRAP")
            for lab, Ps, Pb, yv, stz in ztests:
                for block in BOOTSTRAP_BLOCK_LENGTHS:
                    br = conditional_null_bootstrap_nested(
                        Ps, Pb, yv, start=stz, B=BOOTSTRAP_B,
                        block_len=block, seed=BOOTSTRAP_SEED + 1000 + block
                    )
                    print(
                        f"  {lab} | block {block}: obs CW {br['observed_CW']:.2f}, "
                        f"boot p {br['bootstrap_p']:.4f}, q95 {br['q95']:.2f}, "
                        f"B {br['B_effective']}"
                    )

        if WRITE_DATA_SNAPSHOT:
            snap, manifest = write_data_snapshot(zc, label="gsw_zero_coupon")
            print(f"\nzero-coupon snapshot: {snap}")
            print(f"panel SHA-256: {manifest['sha256_file']}")
            print(f"raw GSW SHA-256: {manifest['gsw_input_sha256_file']}")
        ZERO_COUPON_OK = True
    except Exception as e:
        ZERO_COUPON_OK = False
        print("GSW ZERO-COUPON BRANCH FAILED ->", repr(e))
        print(
            "Upload the frozen original feds200628.csv beside the notebook. "
            "No network request and no GS fallback were attempted."
        )


/tmp/ipykernel_2290/1117197058.py:156: UserWarning: Parsed DataFrame hash differs from the archived pandas hash. The byte-level file hash is valid; this can reflect pandas parsing/version details.
  warnings.warn(


Assembled 599 months: 1976-06-30 -> 2026-05-31
  curve: Frozen Fed GSW zero-coupon (feds200628) | r*: proxy: EWMA real short (hl=120m) | TP ref: THREEFYTP10
  target interpretation: zero-coupon Fama-Bliss | anchor init: post_ewma_bfill

GSW ZERO-COUPON HEADLINE
  basket   OOS +0.0209  CW 1.53  p 0.0629  n 336
  10Y      OOS +0.1553  CW 3.12  p 0.0009  n 336
  10Y TP adds beyond y10: R2 +0.1878  CW 3.70  p 0.0001

CANONICAL CP — annual 1–5Y zero-coupon, paired comparisons
   benchmark      start   n  TP_OOS   TP_CW   B_OOS    B_CW  TP_adds_R2  TP_adds_CW  B_adds_R2  B_adds_CW
canonical CP 1998-04-30 325 +0.0259 +1.5605 -0.0646 +0.1869     +0.0445     +1.4922    -0.0722    -0.1619
   benchmark      start   n  TP_OOS   TP_CW   B_OOS    B_CW  TP_adds_R2  TP_adds_CW  B_adds_R2  B_adds_CW
canonical CP 1998-04-30 325 +0.1782 +3.2011 -0.0306 +0.8992     +0.1914     +2.7323    -0.0654    +0.1590

ZERO-COUPON CONDITIONAL BOOTSTRAP
  ZC basket: mean -> TP_HW | block 12: obs CW 1.53, boot p 0.0610


## Finite-sample inference — recursive conditional-null block bootstrap

This is a **conditional-on-predictors** null bootstrap. The small model generates pseudo-targets,
serially dependent residuals are resampled in circular blocks, and the exact expanding forecast
comparison is re-run in every replication using precomputed recursive OLS weights.

The final archive uses `BOOTSTRAP_B = 9999`, with 12- and 24-month blocks. The test set includes
the marginal basket headline as well as the four long-end identification/benchmark comparisons.
For Cieslak–Povala-style, inference remains conditional on a benchmark constructed from the same
realized-return history and must be described accordingly.


In [38]:

if not DATA_OK or not RUN_CONDITIONAL_BOOTSTRAP:
    print("(conditional bootstrap skipped)")
else:
    CAR, FUT = carry_future_decomposition(Y)
    fut10 = FUT[10]
    cpp_style = cieslak_povala_expanding(Y, rx_bar, pi_trend)
    empty = np.empty((len(rx_bar), 0))

    tests = [
        ("Baseline basket: mean -> TP_HW",
         empty, tpHW_bar, rx_bar, t0),
        ("M1 10Y: y10 -> y10+TP10",
         y10, np.column_stack([y10, tp10]), rx10,
         matched_start(y10, tp10, yv=rx10)),
        ("M1 future: y10 -> y10+TP10",
         y10, np.column_stack([y10, tp10]), fut10,
         matched_start(y10, tp10, yv=fut10)),
        ("M2 10Y: CP-P-style -> +TP10",
         cpp_style, np.column_stack([cpp_style, tp10]), rx10,
         matched_start(cpp_style, tp10, yv=rx10)),
    ]
    if tp_series is not None:
        ext = np.asarray(tp_series, float)
        tests.append((
            "M2 10Y: Kim-Wright -> +TP10",
            ext, np.column_stack([ext, tp10]), rx10,
            matched_start(ext, tp10, yv=rx10),
        ))

    out = []
    for lab, Ps, Pb, yv, st in tests:
        for block in BOOTSTRAP_BLOCK_LENGTHS:
            rr = conditional_null_bootstrap_nested(
                Ps, Pb, yv, start=st, B=BOOTSTRAP_B,
                block_len=block, seed=BOOTSTRAP_SEED + block
            )
            out.append(dict(test=lab, start=str(dates[st].date()), **rr))
            print(
                f"{lab} | block {block}: obs CW {rr['observed_CW']:.2f}, "
                f"boot p {rr['bootstrap_p']:.4f}, q90 {rr['q90']:.2f}, "
                f"q95 {rr['q95']:.2f}, B {rr['B_effective']}"
            )
    bootstrap_table = pd.DataFrame(out)


Baseline basket: mean -> TP_HW | block 12: obs CW 1.37, boot p 0.0856, q90 1.26, q95 1.77, B 9999
Baseline basket: mean -> TP_HW | block 24: obs CW 1.37, boot p 0.0869, q90 1.25, q95 1.78, B 9999
M1 10Y: y10 -> y10+TP10 | block 12: obs CW 3.15, boot p 0.0029, q90 1.20, q95 1.68, B 9999
M1 10Y: y10 -> y10+TP10 | block 24: obs CW 3.15, boot p 0.0027, q90 1.16, q95 1.66, B 9999
M1 future: y10 -> y10+TP10 | block 12: obs CW 2.77, boot p 0.0073, q90 1.21, q95 1.72, B 9999
M1 future: y10 -> y10+TP10 | block 24: obs CW 2.77, boot p 0.0057, q90 1.17, q95 1.67, B 9999
M2 10Y: CP-P-style -> +TP10 | block 12: obs CW 2.29, boot p 0.0274, q90 1.44, q95 1.91, B 9999
M2 10Y: CP-P-style -> +TP10 | block 24: obs CW 2.29, boot p 0.0336, q90 1.54, q95 2.03, B 9999
M2 10Y: Kim-Wright -> +TP10 | block 12: obs CW 1.84, boot p 0.0455, q90 1.28, q95 1.78, B 9999
M2 10Y: Kim-Wright -> +TP10 | block 24: obs CW 1.84, boot p 0.0472, q90 1.31, q95 1.80, B 9999



## Decision rules for the paper

- **Basket:** report its conditional-bootstrap result even if the marginal 10% asymptotic result
  does not survive. Treat it as a conservative descriptive baseline, not the main headline.
- **M1:** report predictive content beyond the contemporaneous level only if the nested 10Y and
  future-component tests remain positive after strict past-only correction and finite-sample
  checks. Do not equate this with identification of a pure structural term premium.
- **M2:** use the suffix **style** for CP and Cieslak–Povala on the GS/CMT branch. Reserve
  “canonical CP” for annual 1–5Y GSW zero-coupon inputs. Show both incremental directions against
  Kim–Wright and label that benchmark pseudo-OOS.
- **Targets:** call GS results **CMT-based Fama–Bliss-style**. Use “zero-coupon Fama–Bliss” only
  for the validated GSW branch.
- **Anchor:** the canonical initialization is `post_ewma_bfill`. Report the old draft ordering as
  an explicit audit specification rather than calling the new version an exact draft replication.
- **Regimes:** report aggregate OOS evidence together with subperiod attribution and channel
  rotation; do not imply uniform superiority.
- **Model object:** use “model-implied term-premium signal” or “macro-anchored excess-yield
  component” and state that the residual may contain model misfit, liquidity and measurement
  error.
- **Bootstrap:** describe inference as conditional on fixed predictors. For the Cieslak–Povala-style
  row, explicitly note that the benchmark itself was estimated from the same realized-return
  history.
- **Reproducibility:** archive both validated raw inputs, generated panels and manifests. No
  economic choice may depend on which website happened to answer.
